# 08 · Optional upstream CHMI CSV → CF/ACDD NetCDF4

This notebook is **independent of the seven-product downstream chain** because public users already receive rain-gauge, temperature and humidity NetCDFs. It calls the confirmed original `chmi_historical_csv_to_netcdf_v5.py` unchanged. It is an optional upstream lesson showing how to download CHMI CSVs and convert them. Default is dry run; never overwrite original downloaded datasets.

In [ ]:
from pathlib import Path
import sys, json
# One shared configuration for ALL notebooks (no per-notebook path editing).
ROOT = Path.cwd().resolve()
if ROOT.name == 'notebooks': ROOT = ROOT.parent
assert (ROOT/'tools_verify_originals.py').is_file(), f'Wrong notebook working folder: {ROOT}'
sys.path.insert(0,str(ROOT/'src'))
from essd_cz.connected import Project
SETTINGS=json.loads((ROOT/'config/active_profile.json').read_text(encoding='utf-8'))
PROFILE=SETTINGS.get('profile','hpc')
if PROFILE not in ('hpc','user'):raise ValueError("Profile must be 'hpc' or 'user'")
CONFIG=ROOT/('config/hpc_source.example.json' if PROFILE=='hpc' else 'config/user_paths.json')
WORKDIR=ROOT/SETTINGS.get('workspace','results/connected_workflow')
project=Project(CONFIG,WORKDIR)
print('Profile:',PROFILE,'Data config:',project.config,'Output:',project.output)


In [ ]:
from essd_cz.station_workflow import *
ORIGINAL_CONVERTER=ROOT/'original/code/chmi/chmi_historical_csv_to_netcdf_v5.py'
print('Original converter:',ORIGINAL_CONVERTER)
print('Use `python',ORIGINAL_CONVERTER,'--help` to confirm options before any downloads')

In [ ]:
# Example only: execute consciously after reading original script's --help.
import subprocess
CATEGORY='humidity'          # or precipitation, temperature
YEARS=[2021]
RESOLUTION='10min'
WORKERS=4
DRY_RUN=True
TARGET_ROOT=project.stage_dir('08_CHMI_source')
cmd=[sys.executable,str(ORIGINAL_CONVERTER),CATEGORY,
     '--resolution',RESOLUTION,'--years',*[str(y) for y in YEARS],
     '--root-dir',str(TARGET_ROOT),'--stage','all','--workers',str(WORKERS)]
print('COMMAND:', ' '.join(cmd))
if not DRY_RUN:
    subprocess.run(cmd,check=True)
else:
    print('Dry run only; no network requests or conversion.')